# 05 — LSTM Model
Sequence-to-one LSTM forecasting silver log-returns.
Benchmarked against ARIMA/MIDAS from previous notebooks.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
plt.rcParams['figure.dpi'] = 120

DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'  # Apple Silicon
print('Device:', DEVICE)

## 1. Load & scale features

In [ ]:
train = pd.read_csv('../data/processed/train.csv', index_col=0, parse_dates=True)
val   = pd.read_csv('../data/processed/val.csv',   index_col=0, parse_dates=True)
test  = pd.read_csv('../data/processed/test.csv',  index_col=0, parse_dates=True)

TARGET   = 'silver_return'
FEATURES = ['silver_return', 'gold_return', 'usd_return', 'copper_return',
            'sp500_return', 'silver_vol_5d', 'silver_vol_21d', 'gs_ratio']
FEATURES = [f for f in FEATURES if f in train.columns]

def clean(df):
    return df[FEATURES].dropna()

tr = clean(train)
va = clean(val)
te = clean(test)

scaler = StandardScaler().fit(tr)
tr_s = scaler.transform(tr)
va_s = scaler.transform(va)
te_s = scaler.transform(te)

target_idx = FEATURES.index(TARGET)
print('Features:', FEATURES)
print('Target index:', target_idx)

## 2. Build sliding-window sequences

In [ ]:
SEQ_LEN = 20  # 20 trading days lookback

def make_sequences(data, seq_len, target_col):
    X, y = [], []
    for i in range(seq_len, len(data)):
        X.append(data[i - seq_len:i])
        y.append(data[i, target_col])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X_tr, y_tr = make_sequences(tr_s, SEQ_LEN, target_idx)
X_va, y_va = make_sequences(va_s, SEQ_LEN, target_idx)
X_te, y_te = make_sequences(te_s, SEQ_LEN, target_idx)

def to_loader(X, y, batch=64, shuffle=True):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y).unsqueeze(1))
    return DataLoader(ds, batch_size=batch, shuffle=shuffle)

train_loader = to_loader(X_tr, y_tr)
val_loader   = to_loader(X_va, y_va, shuffle=False)
test_loader  = to_loader(X_te, y_te, shuffle=False)

print(f'Train seqs: {X_tr.shape}, Val: {X_va.shape}, Test: {X_te.shape}')

## 3. LSTM model

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers,
                            batch_first=True, dropout=dropout)
        self.fc   = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])   # last timestep


model = LSTMForecaster(input_size=len(FEATURES)).to(DEVICE)
print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 4. Train

In [ ]:
EPOCHS   = 50
LR       = 1e-3
PATIENCE = 7

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

train_losses, val_losses = [], []
best_val, patience_cnt  = np.inf, 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    batch_loss = []
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        batch_loss.append(loss.item())

    model.eval()
    with torch.no_grad():
        vl = np.mean([criterion(model(xb.to(DEVICE)), yb.to(DEVICE)).item()
                      for xb, yb in val_loader])
    tl = np.mean(batch_loss)
    train_losses.append(tl)
    val_losses.append(vl)
    scheduler.step(vl)

    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}  train={tl:.6f}  val={vl:.6f}')

    if vl < best_val:
        best_val, patience_cnt = vl, 0
        torch.save(model.state_dict(), '../data/processed/lstm_best.pt')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

# Loss curve
plt.figure(figsize=(8, 3))
plt.plot(train_losses, label='Train')
plt.plot(val_losses,   label='Val')
plt.title('LSTM Training Loss')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Test evaluation

In [ ]:
model.load_state_dict(torch.load('../data/processed/lstm_best.pt'))
model.eval()

preds, actuals = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        p = model(xb.to(DEVICE)).cpu().numpy().flatten()
        preds.extend(p)
        actuals.extend(yb.numpy().flatten())

preds   = np.array(preds)
actuals = np.array(actuals)

rmse = np.sqrt(mean_squared_error(actuals, preds))
mae  = mean_absolute_error(actuals, preds)
da   = np.mean(np.sign(actuals) == np.sign(preds))
print(f'LSTM  RMSE={rmse:.6f}  MAE={mae:.6f}  Dir.Acc={da:.3f}')

pd.DataFrame([{'model': 'LSTM', 'rmse': rmse, 'mae': mae, 'dir_acc': da}]
            ).to_csv('../data/processed/metrics_lstm.csv', index=False)

# Plot
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(actuals[:100], label='Actual', lw=1)
ax.plot(preds[:100],   label='LSTM',   lw=1, alpha=0.8)
ax.set_title('LSTM Forecast vs Actual (first 100 test days)')
ax.legend()
plt.tight_layout()
plt.show()